# Inférence statistique

## Auto-ML

pycaret ne supporte pas encore officiellement Python 3.14.3 donc on utilise à la place le package flaml

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from flaml import AutoML

c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_train = pd.read_csv("../data_finale/featuring/train_featured.csv")
df_val = pd.read_csv("../data_finale/featuring/val_featured.csv")
df_test = pd.read_csv("../data_finale/featuring/test_featured.csv")

In [3]:
colonne_cible = "market_value_in_eur"
colonne_joueur = "player"
colonne_team = "team"
colonne_nation = "nation"

df_train = df_train.dropna(subset=[colonne_cible])
df_val = df_val.dropna(subset=[colonne_cible])
df_test = df_test.dropna(subset=[colonne_cible])

# Séparation des features et de la variable cible
joueurs_test = df_test[colonne_joueur]

# On supprime les colonnes texte et la cible pour l'entraînement
X_train = df_train.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_train = df_train[colonne_cible]

X_val = df_val.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_val = df_val[colonne_cible]

X_test = df_test.drop(columns=[colonne_cible, colonne_joueur, colonne_team, colonne_nation, "position",
                                 "market_value_in_eur_nor"])
y_test = df_test[colonne_cible]

In [4]:
print("--- Entraînement FLAML ---")
automl = AutoML()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    task="regression",
    metric="mae",
    time_budget=300,  # 5 minutes pour FLAML
    estimator_list=["xgboost", "lgbm", "catboost", "rf", "extra_tree", "kneighbor", "enet", "lassolars"],
    seed=1308
)

# Récupération des meilleures prédictions de FLAML
preds_flaml = automl.predict(X_test)
mae_flaml = mean_absolute_error(y_test, preds_flaml)


# Préparation : Sélection dynamique des colonnes sans aucun NaN

# On ne garde que les colonnes où la somme des NaN est égale à 0
colonnes_sans_nan = X_train.columns[X_train.isna().sum() == 0].tolist()

print(f"Filtrage pour les modèles linéaires/SVR :")
print(f" -> {len(colonnes_sans_nan)} colonnes conservées sur {X_train.shape[1]} (0 NaN).")

# Création des sous-ensembles spécifiques "propres"
X_train_sans_nan = X_train[colonnes_sans_nan]
X_test_sans_nan = X_test[colonnes_sans_nan]


print("\n--- Entraînement SVR ---")

# Plus besoin d'imputeur ! On garde juste le StandardScaler (vital pour le SVR)
svr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

# Ajustement de la grille (préfixe svr__ nécessaire à cause du Pipeline)
param_svr = {
    'svr__C': [0.1, 1, 10, 100],
    'svr__kernel': ['rbf', 'linear'],
    'svr__epsilon': [0.01, 0.1, 0.5]
}

grid_svr = GridSearchCV(svr_pipeline, param_svr, scoring='neg_mean_absolute_error', cv=3, n_jobs=-1)
# On entraîne sur les colonnes sans NaN
grid_svr.fit(X_train_sans_nan, y_train)

preds_svr = grid_svr.predict(X_test_sans_nan)
mae_svr = mean_absolute_error(y_test, preds_svr)


print("--- Entraînement Régression Linéaire ---")

# Optionnel mais recommandé : Mettre un StandardScaler ici aussi si vous utilisez 
# des variables aux échelles très différentes, ou laisser LinearRegression brute.
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

# On entraîne également sur les colonnes sans NaN
lr_pipeline.fit(X_train_sans_nan, y_train)

preds_lr = lr_pipeline.predict(X_test_sans_nan)
mae_lr = mean_absolute_error(y_test, preds_lr)



print("\nComparaison des performences (MAE)")
print(f"FLAML : {mae_flaml:,.0f} € (Modèle : {automl.best_estimator})")
print(f"SVR (Optimisé par GridSearch)        : {mae_svr:,.0f} €")
print(f"Régression Linéaire Multiple         : {mae_lr:,.0f} €")

# Trouver le grand gagnant
resultats = {
    "FLAML": (mae_flaml, preds_flaml),
    "SVR": (mae_svr, preds_svr),
    "LinearRegression": (mae_lr, preds_lr)
}
meilleur_approche = min(resultats, key=lambda k: resultats[k][0])
print(f"\nLe meilleur choix final est : {meilleur_approche}")

--- Entraînement FLAML ---
[flaml.automl.logger: 07-01 09:30:51] {2375} INFO - task = regression
[flaml.automl.logger: 07-01 09:30:51] {2383} INFO - Data split method: uniform
[flaml.automl.logger: 07-01 09:30:51] {2386} INFO - Evaluation method: holdout
[flaml.automl.logger: 07-01 09:30:51] {2489} INFO - Minimizing error metric: mae
[flaml.automl.logger: 07-01 09:30:51] {2606} INFO - List of ML learners in AutoML Run: ['xgboost', 'lgbm', 'catboost', 'rf', 'extra_tree', 'kneighbor', 'enet', 'lassolars']
[flaml.automl.logger: 07-01 09:30:51] {2911} INFO - iteration 0, current learner xgboost
[flaml.automl.logger: 07-01 09:30:51] {3046} INFO - Estimated sufficient time budget=1041s. Estimated necessary time budget=3s.
[flaml.automl.logger: 07-01 09:30:51] {3097} INFO -  at 0.4s,	estimator xgboost's best error=9.5835e+06,	best estimator xgboost's best error=9.5835e+06
[flaml.automl.logger: 07-01 09:30:51] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 07-01 09:30:51

In [ ]:
# On extrait le meilleur estimateur trouvé par FLAML
meilleur_modele = automl.model.estimator

# On affiche le nom du modèle pour savoir qui a gagné
print(f"Meilleur modèle trouvé : {type(meilleur_modele).__name__}\n")

# On extrait les hyperparamètres de manière sûre
print("Hyperparamètres finaux :")
if hasattr(meilleur_modele, 'get_all_params'):
    # Si c'est vraiment un CatBoost
    print(meilleur_modele.get_all_params())
else:
    # Si c'est XGBoost, LightGBM ou un autre modèle scikit-learn
    print(meilleur_modele.get_params())

Meilleur modèle trouvé : XGBRegressor

Hyperparamètres finaux :
{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': [], 'colsample_bylevel': 1.0, 'colsample_bynode': None, 'colsample_bytree': 1.0, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': 'lossguide', 'importance_type': None, 'interaction_constraints': None, 'learning_rate': np.float64(0.047039577299965676), 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 0, 'max_leaves': 43, 'min_child_weight': np.float64(4.791541868873825), 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 810, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': None, 'reg_alpha': 0.0009765625, 'reg_lambda': np.float64(1.547191718086146), 'sampling_method': None, 'scale_pos_weight': None, 'subsample': np.fl

## Premiers tests de modélisation

In [ ]:
modeles = {
    
    # Forêts (On garde une bonne profondeur)
    "Random Forest": RandomForestRegressor(
        random_state=1308,
        n_jobs=-1,
        n_estimators=200,
        max_depth=30,
        min_samples_split=5
    ),
    "Extra Trees": ExtraTreesRegressor(
        random_state=1308,
        n_jobs=-1,
        n_estimators=200,
        max_depth=30,
        min_samples_split=5
    ),
    
    # Boosting (Alignés sur la stratégie gagnante de FLAML)
    "XGBoost": XGBRegressor(
    random_state=1308,
    n_jobs=-1,
    
    # Paramètres de structure ajustés par FLAML
    tree_method='hist',
    grow_policy='lossguide',
    max_depth=0,                # Aucune limite de profondeur
    max_leaves=43,              # car on contrôle la complexité par le nombre de feuilles
    enable_categorical=True,    
    
    # Paramètres de régularisation optimisés
    min_child_weight=4.791541868873825,
    subsample=0.7386565774212087,
    reg_alpha=0.0009765625,
    reg_lambda=1.547191718086146,
    
    # Stratégie d'apprentissage
    learning_rate=0.01,         # Plus bas et robuste
    n_estimators=3800,          # Augmenté proportionnellement pour compenser la baisse du learning rate
    
    # Paramètres généraux
    objective='reg:squarederror',
    verbosity=0
),
    "LightGBM": LGBMRegressor(
        random_state=1308,
        n_jobs=-1,
        n_estimators=3000,       # Les arbres de LightGBM sont très rapides à construire
        learning_rate=0.01,
        max_depth=6,
        verbose=-1
    ),
    "CatBoost": CatBoostRegressor(
        random_state=1308,
        iterations=8192,
        learning_rate=0.005,
        depth=6,
        l2_leaf_reg=3,
        subsample=0.8,
        early_stopping_rounds=100,
        verbose=0                 
    ),
}

In [8]:
# Entraînement puis évaluation
resultats = {}

for nom, modele in modeles.items():
    print(f"Entraînement de {nom}...")
    
    # Entraînement sur le jeu de train uniquement
    modele.fit(X_train, y_train)
    
    # Prédictions
    preds_val = modele.predict(X_val)
    
    mae_val = mean_absolute_error(y_val, preds_val)
    r2_val = r2_score(y_val, preds_val)
    rmse_val = np.sqrt(mean_squared_error(y_val, preds_val))
    mape_val = mean_absolute_percentage_error(y_val, preds_val)

    # R² Ajusté
    n = len(y_val)          # Nombre d'observations
    p = X_val.shape[1]      # Nombre de variables (colonnes)
    r2_ajuste_val = 1 - (1 - r2_val) * (n - 1) / (n - p - 1)
    
    # Sauvegarde pour le tableau final
    resultats[nom] = {
        "MAE Val": mae_val,
        "RMSE Val": rmse_val,
        "MAPE Val": mape_val,
        "R² Val": r2_val,
        "R² Ajusté Val": r2_ajuste_val
    }
    
    print(f"   -> Validation | MAE : {mae_val:,.0f} € | RMSE : {rmse_val:,.0f} € | MAPE : {mape_val:.2%} | R² : {r2_val:.2%} | R² Ajusté : {r2_ajuste_val:.2%}" )

# Comparaion des modèles
print("CLASSEMENT FINAL (Trié par la plus petite erreur sur test)")
tableau_resultats = []
for nom, metrics in resultats.items():
    tableau_resultats.append({
        "Modèle": nom,
        "Erreur moyenne Val (MAE)": f"{metrics['MAE Val']:,.0f} €",
        "Score R² Val": f"{metrics['R² Val']:.2%}",
        "Score R² Ajusté Val": f"{metrics['R² Ajusté Val']:.2%}"
    })

# En régression, le meilleur modèle est celui avec la MAE la plus basse
df_final = pd.DataFrame(tableau_resultats).sort_values(by="Erreur moyenne Val (MAE)", ascending=True)
print(df_final.to_string(index=False))

Entraînement de Random Forest...
   -> Validation | MAE : 5,040,843 € | RMSE : 9,713,330 € | MAPE : 89.33% | R² : 70.90%
Entraînement de Extra Trees...
   -> Validation | MAE : 4,983,436 € | RMSE : 9,540,966 € | MAPE : 91.67% | R² : 71.92%
Entraînement de XGBoost...
   -> Validation | MAE : 4,612,237 € | RMSE : 8,945,683 € | MAPE : 81.67% | R² : 75.31%
Entraînement de LightGBM...
   -> Validation | MAE : 4,782,923 € | RMSE : 9,010,973 € | MAPE : 88.97% | R² : 74.95%
Entraînement de CatBoost...
   -> Validation | MAE : 4,659,503 € | RMSE : 8,835,834 € | MAPE : 91.54% | R² : 75.92%
CLASSEMENT FINAL (Trié par la plus petite erreur sur test)
       Modèle Erreur moyenne Val (MAE) Score R² Val
      XGBoost              4,612,237 €       75.31%
     CatBoost              4,659,503 €       75.92%
     LightGBM              4,782,923 €       74.95%
  Extra Trees              4,983,436 €       71.92%
Random Forest              5,040,843 €       70.90%


In [ ]:
# Entraînement puis évaluation
resultats = {}

for nom, modele in modeles.items():
    print(f"Entraînement de {nom}...")
    
    # Entraînement sur le jeu de train uniquement
    modele.fit(X_train, y_train)
    
    # Prédictions
    preds_test = modele.predict(X_test)
    
    mae_test = mean_absolute_error(y_test, preds_test)
    r2_test = r2_score(y_test, preds_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, preds_test))
    mape_test = mean_absolute_percentage_error(y_test, preds_test)

    # R² Ajusté
    n = len(y_test)          # Nombre d'observations
    p = X_test.shape[1]      # Nombre de variables (colonnes)
    r2_ajuste_test = 1 - (1 - r2_test) * (n - 1) / (n - p - 1)
    
    # Sauvegarde pour le tableau final
    resultats[nom] = {
        "MAE Test": mae_test,
        "R² Test": r2_test,
        "MAPE Test": mape_test,
        "R² Test": r2_test,
        "R² Ajusté Test": r2_ajuste_test
    }
    
    print(f"   -> Test | MAE : {mae_test:,.0f} € | RMSE : {rmse_test:,.0f} € | MAPE : {mape_test:.2%} | R² : {r2_test:.2%} | R² Ajusté : {r2_ajuste_test:.2%}" )

# Comparaion des modèles
print("CLASSEMENT FINAL (Trié par la plus petite erreur sur test)")
tableau_resultats = []
for nom, metrics in resultats.items():
    tableau_resultats.append({
        "Modèle": nom,
        "Erreur moyenne Test (MAE)": f"{metrics['MAE Test']:,.0f} €",
        "Score R² Test": f"{metrics['R² Test']:.2%}",
        "Score R² Ajusté Test": f"{metrics['R² Ajusté Test']:.2%}"
    })

# En régression, le meilleur modèle est celui avec la MAE la plus basse
df_final = pd.DataFrame(tableau_resultats).sort_values(by="Erreur moyenne Test (MAE)", ascending=True)
print(df_final.to_string(index=False))

Entraînement de Random Forest...
   -> Test | MAE : 5,378,551 € | RMSE : 10,630,318 € | MAPE : 80.18% | R² : 67.69% | R² Ajusté : 65.18%
Entraînement de Extra Trees...
   -> Test | MAE : 5,331,168 € | RMSE : 10,490,978 € | MAPE : 83.06% | R² : 68.53% | R² Ajusté : 66.08%
Entraînement de XGBoost...
   -> Test | MAE : 4,977,951 € | RMSE : 9,748,343 € | MAPE : 77.20% | R² : 72.83% | R² Ajusté : 70.72%
Entraînement de LightGBM...
   -> Test | MAE : 5,073,985 € | RMSE : 9,701,387 € | MAPE : 79.53% | R² : 73.09% | R² Ajusté : 71.00%
Entraînement de CatBoost...
   -> Test | MAE : 5,062,656 € | RMSE : 9,765,905 € | MAPE : 78.63% | R² : 72.73% | R² Ajusté : 70.61%
CLASSEMENT FINAL (Trié par la plus petite erreur sur test)
       Modèle Erreur moyenne Test (MAE) Score R² Test Score R² Ajusté Test
      XGBoost               4,977,951 €        72.83%               70.72%
     CatBoost               5,062,656 €        72.73%               70.61%
     LightGBM               5,073,985 €        73.09

## Tuning des hyperparamètres (Optuna)

On cherche, pour chaque modèle, les hyperparamètres qui **minimisent le MAE sur le jeu de validation** (`X_val` / `y_val`). Le jeu de **test reste intouché** pendant tout le tuning : il ne sert qu'à l'évaluation finale, une seule fois, pour ne pas biaiser l'estimation de la performance réelle.

On utilise [Optuna](https://optuna.org/) (recherche bayésienne / TPE), beaucoup plus efficace qu'un `GridSearchCV` classique pour ce genre d'espaces de recherche (moins d'essais nécessaires pour un résultat équivalent ou meilleur).

In [10]:
# Installation si nécessaire (à décommenter la première fois)
# !pip install optuna

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 40          # nombre d'essais par modèle : augmentez si vous avez du temps (ex: 100-200)
RANDOM_STATE = 1308

# Sous-ensemble sans NaN pour la validation (nécessaire pour SVR / modèles linéaires)
X_val_sans_nan = X_val[colonnes_sans_nan]


In [11]:
def tune_forest(model_class, trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8, 1.0]),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }
    model = model_class(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return mean_absolute_error(y_val, preds)

print("--- Tuning Random Forest ---")
study_rf = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(lambda t: tune_forest(RandomForestRegressor, t), n_trials=N_TRIALS, show_progress_bar=True)
print("Meilleur MAE :", study_rf.best_value)
print("Meilleurs paramètres :", study_rf.best_params)

print("\n--- Tuning Extra Trees ---")
study_et = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_et.optimize(lambda t: tune_forest(ExtraTreesRegressor, t), n_trials=N_TRIALS, show_progress_bar=True)
print("Meilleur MAE :", study_et.best_value)
print("Meilleurs paramètres :", study_et.best_params)


--- Tuning Random Forest ---


Best trial: 34. Best value: 5.00757e+06: 100%|██████████| 40/40 [10:28<00:00, 15.70s/it]


Meilleur MAE : 5007569.87509529
Meilleurs paramètres : {'n_estimators': 1000, 'max_depth': 25, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 0.8}

--- Tuning Extra Trees ---


Best trial: 36. Best value: 4.94011e+06: 100%|██████████| 40/40 [08:29<00:00, 12.74s/it]

Meilleur MAE : 4940109.586605793
Meilleurs paramètres : {'n_estimators': 1000, 'max_depth': 36, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 1.0}


In [12]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 5000, step=250),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
        "tree_method": "hist",
        "enable_categorical": True,
        "objective": "reg:squarederror",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": 0,
    }
    model = XGBRegressor(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return mean_absolute_error(y_val, preds)

print("--- Tuning XGBoost ---")
study_xgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
print("Meilleur MAE :", study_xgb.best_value)
print("Meilleurs paramètres :", study_xgb.best_params)


--- Tuning XGBoost ---


Best trial: 33. Best value: 4.61382e+06: 100%|██████████| 40/40 [20:31<00:00, 30.79s/it]

Meilleur MAE : 4613818.206812449
Meilleurs paramètres : {'n_estimators': 1750, 'max_depth': 10, 'learning_rate': 0.008495483131938328, 'subsample': 0.5134049773318177, 'colsample_bytree': 0.6522655023903593, 'min_child_weight': 9.375693632459882, 'reg_alpha': 0.028715173071664254, 'reg_lambda': 0.016463364302549705}


In [13]:
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 5000, step=250),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
    }
    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return mean_absolute_error(y_val, preds)

print("--- Tuning LightGBM ---")
study_lgbm = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS, show_progress_bar=True)
print("Meilleur MAE :", study_lgbm.best_value)
print("Meilleurs paramètres :", study_lgbm.best_params)


--- Tuning LightGBM ---


Best trial: 30. Best value: 4.68754e+06: 100%|██████████| 40/40 [06:26<00:00,  9.66s/it]

Meilleur MAE : 4687541.175203909
Meilleurs paramètres : {'n_estimators': 1250, 'num_leaves': 55, 'max_depth': 15, 'learning_rate': 0.01576832194916594, 'subsample': 0.9746109641661995, 'colsample_bytree': 0.6363035291373982, 'reg_alpha': 0.0702478621553085, 'reg_lambda': 0.002020284753715106}


In [14]:
def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 1000, 10000, step=500),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "bootstrap_type": "Bernoulli",   # nécessaire pour pouvoir utiliser 'subsample'
        "random_state": RANDOM_STATE,
        "early_stopping_rounds": 100,
        "verbose": 0,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
    preds = model.predict(X_val)
    return mean_absolute_error(y_val, preds)

print("--- Tuning CatBoost ---")
study_cat = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_cat.optimize(objective_catboost, n_trials=N_TRIALS, show_progress_bar=True)
print("Meilleur MAE :", study_cat.best_value)
print("Meilleurs paramètres :", study_cat.best_params)


--- Tuning CatBoost ---


Best trial: 35. Best value: 4.5644e+06: 100%|██████████| 40/40 [1:07:12<00:00, 100.81s/it] 

Meilleur MAE : 4564400.014541649
Meilleurs paramètres : {'iterations': 3500, 'depth': 7, 'learning_rate': 0.021674111348623584, 'l2_leaf_reg': 1.5396241332040423, 'subsample': 0.8069489117199329}


In [15]:
from sklearn.linear_model import ElasticNet

def objective_svr(trial):
    params = {
        "C": trial.suggest_float("C", 0.01, 100, log=True),
        "epsilon": trial.suggest_float("epsilon", 0.001, 1, log=True),
        "kernel": trial.suggest_categorical("kernel", ["rbf", "linear"]),
        "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),
    }
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(**params))
    ])
    pipeline.fit(X_train_sans_nan, y_train)
    preds = pipeline.predict(X_val_sans_nan)
    return mean_absolute_error(y_val, preds)

print("--- Tuning SVR ---")
study_svr = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_svr.optimize(objective_svr, n_trials=N_TRIALS, show_progress_bar=True)
print("Meilleur MAE :", study_svr.best_value)
print("Meilleurs paramètres :", study_svr.best_params)


def objective_enet(trial):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-4, 10, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "random_state": RANDOM_STATE,
        "max_iter": 10000,
    }
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("enet", ElasticNet(**params))
    ])
    pipeline.fit(X_train_sans_nan, y_train)
    preds = pipeline.predict(X_val_sans_nan)
    return mean_absolute_error(y_val, preds)

print("\n--- Tuning ElasticNet (remplace la régression linéaire simple, qui n'a pas d'hyperparamètre à tuner) ---")
study_enet = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_enet.optimize(objective_enet, n_trials=N_TRIALS, show_progress_bar=True)
print("Meilleur MAE :", study_enet.best_value)
print("Meilleurs paramètres :", study_enet.best_params)


--- Tuning SVR ---


Best trial: 15. Best value: 8.5774e+06: 100%|██████████| 40/40 [04:50<00:00,  7.26s/it] 


Meilleur MAE : 8577403.359818045
Meilleurs paramètres : {'C': 99.4615861939721, 'epsilon': 0.02020898211772718, 'kernel': 'linear', 'gamma': 'auto'}

--- Tuning ElasticNet (remplace la régression linéaire simple, qui n'a pas d'hyperparamètre à tuner) ---


Best trial: 11. Best value: 6.5643e+06: 100%|██████████| 40/40 [02:48<00:00,  4.21s/it] 

Meilleur MAE : 6564297.797945763
Meilleurs paramètres : {'alpha': 9.313014897989799, 'l1_ratio': 0.988539726692843}


### Modèles finaux avec hyperparamètres tunés

In [16]:
modeles_tuned = {
    "Random Forest": RandomForestRegressor(**study_rf.best_params, random_state=RANDOM_STATE, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(**study_et.best_params, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBRegressor(**study_xgb.best_params, tree_method="hist", enable_categorical=True,
                             objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
    "LightGBM": LGBMRegressor(**study_lgbm.best_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostRegressor(**study_cat.best_params, bootstrap_type="Bernoulli",
                                   random_state=RANDOM_STATE, early_stopping_rounds=100, verbose=0),
}

# Modèles qui utilisent le sous-ensemble de colonnes sans NaN (SVR, ElasticNet)
modeles_tuned_sans_nan = {
    "SVR": Pipeline([("scaler", StandardScaler()), ("svr", SVR(**study_svr.best_params))]),
    "ElasticNet": Pipeline([("scaler", StandardScaler()),
                             ("enet", ElasticNet(**study_enet.best_params, random_state=RANDOM_STATE, max_iter=10000))]),
}


In [17]:
def evaluer(modele, X_tr, y_tr, X_ev, y_ev):
    modele.fit(X_tr, y_tr)
    preds = modele.predict(X_ev)
    mae = mean_absolute_error(y_ev, preds)
    rmse = np.sqrt(mean_squared_error(y_ev, preds))
    mape = mean_absolute_percentage_error(y_ev, preds)
    r2 = r2_score(y_ev, preds)
    n, p = len(y_ev), X_ev.shape[1]
    r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2, "R2 Ajusté": r2_adj}

# --- Évaluation sur la VALIDATION (sert à vérifier le gain du tuning) ---
resultats_tuned_val = {}
for nom, modele in modeles_tuned.items():
    print(f"Entraînement (tuné) de {nom}...")
    resultats_tuned_val[nom] = evaluer(modele, X_train, y_train, X_val, y_val)

for nom, modele in modeles_tuned_sans_nan.items():
    print(f"Entraînement (tuné) de {nom}...")
    resultats_tuned_val[nom] = evaluer(modele, X_train_sans_nan, y_train, X_val_sans_nan, y_val)

df_tuned_val = pd.DataFrame(resultats_tuned_val).T.sort_values("MAE")
print("\nCLASSEMENT (Validation, modèles tunés)")
print(df_tuned_val)


Entraînement (tuné) de Random Forest...
Entraînement (tuné) de Extra Trees...
Entraînement (tuné) de XGBoost...
Entraînement (tuné) de LightGBM...
Entraînement (tuné) de CatBoost...
Entraînement (tuné) de SVR...
Entraînement (tuné) de ElasticNet...

CLASSEMENT (Validation, modèles tunés)
                        MAE          RMSE      MAPE        R2  R2 Ajusté
CatBoost       4.560335e+06  8.811370e+06  0.838286  0.760500   0.743030
XGBoost        4.613818e+06  9.003502e+06  0.827792  0.749942   0.731701
LightGBM       4.687541e+06  9.112240e+06  0.820223  0.743865   0.725182
Extra Trees    4.940110e+06  9.510305e+06  0.894550  0.720998   0.700647
Random Forest  5.007570e+06  9.729524e+06  0.886730  0.707988   0.686687
ElasticNet     6.564298e+06  1.122869e+07  2.288003  0.611066   0.585626
SVR            8.577403e+06  1.775098e+07  1.156699  0.028008  -0.035568


In [18]:
# --- Évaluation finale sur le TEST (à ne lancer qu'une seule fois, à la toute fin) ---
resultats_tuned_test = {}
for nom, modele in modeles_tuned.items():
    resultats_tuned_test[nom] = evaluer(modele, X_train, y_train, X_test, y_test)

for nom, modele in modeles_tuned_sans_nan.items():
    resultats_tuned_test[nom] = evaluer(modele, X_train_sans_nan, y_train, X_test_sans_nan, y_test)

df_tuned_test = pd.DataFrame(resultats_tuned_test).T.sort_values("MAE")
print("CLASSEMENT FINAL (Test, modèles tunés)")
print(df_tuned_test)


CLASSEMENT FINAL (Test, modèles tunés)
                        MAE          RMSE      MAPE        R2  R2 Ajusté
CatBoost       4.999971e+06  9.776233e+06  0.766244  0.726734   0.705476
LightGBM       5.029998e+06  9.830999e+06  0.749376  0.723664   0.702167
XGBoost        5.051068e+06  9.913462e+06  0.760559  0.719009   0.697149
Extra Trees    5.280507e+06  1.051981e+07  0.808601  0.683585   0.658969
Random Forest  5.335912e+06  1.066077e+07  0.785867  0.675048   0.649769
ElasticNet     7.198718e+06  1.226207e+07  2.142806  0.570098   0.540123
SVR            9.362710e+06  1.878492e+07  1.112459 -0.008930  -0.079277
